<a href="https://colab.research.google.com/github/maraymtaher/Detection_P/blob/main/PFE_Desertification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rasterio --quiet
!pip install gdal --quiet
!pip install earthpy --quiet

print("installation terminée")


installation terminée


In [ ]:
# 1 : Monter Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 2 : Importer les bibliothèques nécessaires

import zipfile   # Lecture et extraction des archives ZIP
import os        # Navigation dans le système de fichiers
from pathlib import Path  # Gestion propre des chemins de fichiers
import shutil    # Opérations sur fichiers et dossiers
print("✅ Bibliothèques importées.")

✅ Bibliothèques importées.


In [ ]:
# 3 : Définir le chemin vers ton dossier Sentinel-2
DOSSIER_DRIVE = "/content/drive/MyDrive/Mn_projet_PFE/images_sent-2"  # Racine de ton Drive

# Afficher le contenu de la racine de ton Drive
print("📁 Contenu de la racine de ton Google Drive :\n")
for element in sorted(os.listdir(DOSSIER_DRIVE)):
    chemin_complet = os.path.join(DOSSIER_DRIVE, element)
    if os.path.isdir(chemin_complet):
        print(f"  📂 [Dossier] {element}")
    else:
        taille = os.path.getsize(chemin_complet) / (1024 * 1024)  # en Mo
        print(f"  📄 [Fichier] {element}  —  {taille:.2f} Mo")

📁 Contenu de la racine de ton Google Drive :

  📄 [Fichier] 2017_02.zip  —  1104.68 Mo
  📄 [Fichier] 2018_02.zip  —  1112.21 Mo
  📄 [Fichier] 2019_02.zip  —  1114.11 Mo
  📄 [Fichier] 2020_02.zip  —  1091.24 Mo
  📄 [Fichier] 2021_02.zip  —  1117.28 Mo
  📄 [Fichier] 2022_02.zip  —  1137.49 Mo
  📄 [Fichier] 2023_02.zip  —  1137.49 Mo
  📄 [Fichier] 2024_02.zip  —  1106.55 Mo
  📄 [Fichier] 2025_02.zip  —  1111.84 Mo
  📄 [Fichier] 2026_02.zip  —  1102.67 Mo


In [ ]:
# 4 : Définir le chemin exact vers tes données Sentinel-2

CHEMIN_SENTINEL = "/content/drive/MyDrive/Mn_projet_PFE/images_sent-2/"

# Vérifier que le dossier existe
if os.path.exists(CHEMIN_SENTINEL):
    print(f"✅ Dossier trouvé : {CHEMIN_SENTINEL}\n")
else:
    print(f"❌ Dossier introuvable : {CHEMIN_SENTINEL}")
    print("   Vérifie le nom et le chemin de ton dossier dans Drive.")

✅ Dossier trouvé : /content/drive/MyDrive/Mn_projet_PFE/images_sent-2/



In [ ]:
# 5 : Lister et analyser les fichiers ZIP Sentinel-2
# Cette cellule liste tous les fichiers ZIP présents dans ton dossier.

fichiers_zip = []

print(" Fichiers ZIP Sentinel-2 détectés :\n")

for fichier in sorted(os.listdir(CHEMIN_SENTINEL)):
    if fichier.endswith(".zip"):
        chemin_fichier = os.path.join(CHEMIN_SENTINEL, fichier)
        taille_mo = os.path.getsize(chemin_fichier) / (1024 * 1024)
        fichiers_zip.append(fichier)
        print(f"  ✅ {fichier}  —  {taille_mo:.1f} Mo")

print(f"\n Nombre total de fichiers ZIP : {len(fichiers_zip)}")

if len(fichiers_zip) == 0:
    print("\n Aucun fichier ZIP trouvé.")
    print("   Vérifie que tes fichiers sont bien dans le dossier indiqué.")
    print("   Vérifie aussi que les fichiers ont bien l'extension .zip (minuscule).")

 Fichiers ZIP Sentinel-2 détectés :

  ✅ 2017_02.zip  —  1104.7 Mo
  ✅ 2018_02.zip  —  1112.2 Mo
  ✅ 2019_02.zip  —  1114.1 Mo
  ✅ 2020_02.zip  —  1091.2 Mo
  ✅ 2021_02.zip  —  1117.3 Mo
  ✅ 2022_02.zip  —  1137.5 Mo
  ✅ 2023_02.zip  —  1137.5 Mo
  ✅ 2024_02.zip  —  1106.6 Mo
  ✅ 2025_02.zip  —  1111.8 Mo
  ✅ 2026_02.zip  —  1102.7 Mo

 Nombre total de fichiers ZIP : 10


In [ ]:
# 6 : Analyser les noms des fichiers pour extraire les dates
# Les fichiers Sentinel-2 ont un nom standardisé qui contient la date d'acquisition.
# Exemple : S2A_MSIL2A_20220215T094031_..._.SAFE.zip
# La date est encodée sous la forme YYYYMMDD.

import re

print(" Analyse des dates d'acquisition :\n")

dates_detectees = []

for fichier in fichiers_zip:
    # Recherche du motif de date dans le nom du fichier
    # Format Sentinel-2 : YYYYMMDDTHHMMSS
    correspondance = re.search(r'(\d{8})T\d{6}', fichier)
    if correspondance:
        date_brute = correspondance.group(1)
        annee = date_brute[:4]
        mois = date_brute[4:6]
        jour = date_brute[6:8]
        date_lisible = f"{jour}/{mois}/{annee}"
        dates_detectees.append(date_lisible)
        print(f"   {fichier[:50]}...  →  Date : {date_lisible}")
    else:
        print(f"   Date non détectée dans : {fichier}")

print(f"\n Nombre d'images avec date identifiée : {len(dates_detectees)}")

 Analyse des dates d'acquisition :

   Date non détectée dans : 2017_02.zip
   Date non détectée dans : 2018_02.zip
   Date non détectée dans : 2019_02.zip
   Date non détectée dans : 2020_02.zip
   Date non détectée dans : 2021_02.zip
   Date non détectée dans : 2022_02.zip
   Date non détectée dans : 2023_02.zip
   Date non détectée dans : 2024_02.zip
   Date non détectée dans : 2025_02.zip
   Date non détectée dans : 2026_02.zip

 Nombre d'images avec date identifiée : 0


In [ ]:
#  Définition des chemins de travail

CHEMIN_SENTINEL = "/content/drive/MyDrive/Mn_projet_PFE/images_sent-2"

# Dossier de destination pour les fichiers extraits (local à Colab)
CHEMIN_EXTRACTION = "/content/sentinel-2_extraits"

# Création du dossier d'extraction s'il n'existe pas déjà
# exist_ok=True signifie : ne pas générer d'erreur si le dossier existe déjà
os.makedirs(CHEMIN_EXTRACTION, exist_ok=True)

print(f" Source des ZIP       : {CHEMIN_SENTINEL}")
print(f" Dossier d'extraction : {CHEMIN_EXTRACTION}")
print("✅ Dossiers configurés.")


 Source des ZIP       : /content/drive/MyDrive/Mn_projet_PFE/images_sent-2
 Dossier d'extraction : /content/sentinel-2_extraits
✅ Dossiers configurés.


In [ ]:
#  Récupération de la liste des fichiers ZIP
# On liste tous les fichiers .zip présents dans mon dossier Drive.
# On les trie pour les traiter dans l'ordre chronologique.

# Lister tous les fichiers ZIP et les trier alphabétiquement
# (ce qui correspond ici à l'ordre chronologique : 2017, 2018, ...)
liste_zip = sorted([
    f for f in os.listdir(CHEMIN_SENTINEL)
    if f.endswith(".zip")   # On garde uniquement les fichiers .zip
])

print(f"\n Fichiers ZIP détectés ({len(liste_zip)} au total) :\n")

# Afficher chaque fichier avec sa taille
for nom_zip in liste_zip:
    chemin_complet = os.path.join(CHEMIN_SENTINEL, nom_zip)
    taille_mo = os.path.getsize(chemin_complet) / (1024 * 1024)
    print(f"  ✅ {nom_zip}  —  {taille_mo:.1f} Mo")


 Fichiers ZIP détectés (10 au total) :

  ✅ 2017_02.zip  —  1104.7 Mo
  ✅ 2018_02.zip  —  1112.2 Mo
  ✅ 2019_02.zip  —  1114.1 Mo
  ✅ 2020_02.zip  —  1091.2 Mo
  ✅ 2021_02.zip  —  1117.3 Mo
  ✅ 2022_02.zip  —  1137.5 Mo
  ✅ 2023_02.zip  —  1137.5 Mo
  ✅ 2024_02.zip  —  1106.6 Mo
  ✅ 2025_02.zip  —  1111.8 Mo
  ✅ 2026_02.zip  —  1102.7 Mo


In [ ]:
#  Exploration du contenu d'un ZIP sans l'extraire


# zipfile.ZipFile() ouvre le ZIP en lecture sans l'extraire.
# namelist() retourne la liste de tous les fichiers à l'intérieur.

if len(liste_zip) > 0:

    #  prend le premier fichier ZIP pour l'explorer
    premier_zip = os.path.join(CHEMIN_SENTINEL, liste_zip[0])

    print(f"\n🔍 Exploration du contenu de : {liste_zip[0]}\n")

    # Ouverture du ZIP en mode lecture ('r' = read)
    with zipfile.ZipFile(premier_zip, 'r') as archive:

        # Récupération de la liste de tous les fichiers dans l'archive
        contenu = archive.namelist()

        print(f"   Nombre total de fichiers dans cette archive : {len(contenu)}\n")

        # Afficher les 30 premiers fichiers pour avoir un aperçu
        print("   Aperçu des premiers fichiers (30 premiers) :\n")
        for chemin_fichier in contenu[:30]:
            print(f"    {chemin_fichier}")

        print("\n  ... (liste tronquée pour lisibilité)")


🔍 Exploration du contenu de : 2017_02.zip

   Nombre total de fichiers dans cette archive : 95

   Aperçu des premiers fichiers (30 premiers) :

    S2A_MSIL2A_20170228T092021_N0500_R093_T33PVQ_20230926T044000.SAFE/DATASTRIP/DS_S2RP_20230926T044000_S20170228T092713/MTD_DS.xml
    S2A_MSIL2A_20170228T092021_N0500_R093_T33PVQ_20230926T044000.SAFE/DATASTRIP/DS_S2RP_20230926T044000_S20170228T092713/QI_DATA/FORMAT_CORRECTNESS.xml
    S2A_MSIL2A_20170228T092021_N0500_R093_T33PVQ_20230926T044000.SAFE/DATASTRIP/DS_S2RP_20230926T044000_S20170228T092713/QI_DATA/GENERAL_QUALITY.xml
    S2A_MSIL2A_20170228T092021_N0500_R093_T33PVQ_20230926T044000.SAFE/DATASTRIP/DS_S2RP_20230926T044000_S20170228T092713/QI_DATA/GEOMETRIC_QUALITY.xml
    S2A_MSIL2A_20170228T092021_N0500_R093_T33PVQ_20230926T044000.SAFE/DATASTRIP/DS_S2RP_20230926T044000_S20170228T092713/QI_DATA/RADIOMETRIC_QUALITY.xml
    S2A_MSIL2A_20170228T092021_N0500_R093_T33PVQ_20230926T044000.SAFE/DATASTRIP/DS_S2RP_20230926T044000_S20170228T092

In [ ]:
#  Recherche des bandes spectrales importantes

# On cherche spécifiquement les fichiers correspondant aux bandes
# B02, B03, B04 et B08 dans le dossier R10m (résolution 10 mètres).
#
# Les fichiers de bandes Sentinel-2 ont l'extension .jp2 (JPEG 2000).
# Leur nom contient le code de la bande : _B02_, _B03_, etc.

if len(liste_zip) > 0:

    premier_zip = os.path.join(CHEMIN_SENTINEL, liste_zip[0])

    # Les bandes dont on a besoin pour ce projet
    bandes_cibles = ["B02", "B03", "B04", "B08"]

    print("\n Recherche des bandes spectrales nécessaires (B02, B03, B04, B08) :\n")

    with zipfile.ZipFile(premier_zip, 'r') as archive:

        contenu = archive.namelist()

        # Parcourir tous les fichiers de l'archive
        for fichier in contenu:

            #  cherche uniquement les fichiers .jp2 dans le dossier R10m
            # car c'est là que sont les bandes à 10 mètres de résolution
            if "R10m" in fichier and fichier.endswith(".jp2"):

                # Vérifier si ce fichier correspond à une de nos bandes cibles
                for bande in bandes_cibles:
                    if f"_{bande}_" in fichier:
                        print(f"  ✅ {bande} trouvée : {fichier}")


 Recherche des bandes spectrales nécessaires (B02, B03, B04, B08) :

  ✅ B02 trouvée : S2A_MSIL2A_20170228T092021_N0500_R093_T33PVQ_20230926T044000.SAFE/GRANULE/L2A_T33PVQ_A008813_20170228T092713/IMG_DATA/R10m/T33PVQ_20170228T092021_B02_10m.jp2
  ✅ B03 trouvée : S2A_MSIL2A_20170228T092021_N0500_R093_T33PVQ_20230926T044000.SAFE/GRANULE/L2A_T33PVQ_A008813_20170228T092713/IMG_DATA/R10m/T33PVQ_20170228T092021_B03_10m.jp2
  ✅ B04 trouvée : S2A_MSIL2A_20170228T092021_N0500_R093_T33PVQ_20230926T044000.SAFE/GRANULE/L2A_T33PVQ_A008813_20170228T092713/IMG_DATA/R10m/T33PVQ_20170228T092021_B04_10m.jp2
  ✅ B08 trouvée : S2A_MSIL2A_20170228T092021_N0500_R093_T33PVQ_20230926T044000.SAFE/GRANULE/L2A_T33PVQ_A008813_20170228T092713/IMG_DATA/R10m/T33PVQ_20170228T092021_B08_10m.jp2


In [ ]:

#  Extraction complète de tous les fichiers ZIP

# Maintenant qu'on a compris la structure, on extrait tous les ZIP.

# Pour chaque fichier ZIP :
#   1. On crée un sous-dossier nommé d'après l'année (ex: "2017")
#   2. On extrait tout le contenu dans ce sous-dossier
#
# Cela permet de garder une organisation claire par année.
# ------------------------------------------------------------

print("\n Début de l'extraction des archives ZIP...\n")
print(" Cette opération peut prendre plusieurs minutes selon la taille des fichiers.\n")

# Dictionnaire pour stocker les chemins d'extraction par année
chemins_extraits = {}

for nom_zip in liste_zip:

    # Construire le chemin complet vers ce fichier ZIP dans Drive
    chemin_zip = os.path.join(CHEMIN_SENTINEL, nom_zip)

    # Extraire l'année depuis le nom du fichier
    # Ex : "2017_02.zip" → on prend les 4 premiers caractères → "2017"
    annee = nom_zip[:4]

    # Créer le dossier de destination pour cette année
    # Ex : /content/sentinel2_extraits/2017/
    dossier_destination = os.path.join(CHEMIN_EXTRACTION, annee)
    os.makedirs(dossier_destination, exist_ok=True)

    # Vérifier si ce fichier a déjà été extrait (pour éviter de refaire)
    # On vérifie si le dossier de destination contient déjà des fichiers
    if len(os.listdir(dossier_destination)) > 0:
        print(f"    {nom_zip} déjà extrait — on passe.")
        chemins_extraits[annee] = dossier_destination
        continue  # Passer au fichier suivant

    # Extraction du ZIP dans le dossier de destination
    print(f"   Extraction de {nom_zip} → {dossier_destination} ...")

    with zipfile.ZipFile(chemin_zip, 'r') as archive:
        # extractall() extrait tous les fichiers dans le dossier indiqué
        archive.extractall(dossier_destination)

    # Calculer la taille extraite
    taille_extraite = sum(
        os.path.getsize(os.path.join(racine, f))
        for racine, dossiers, fichiers in os.walk(dossier_destination)
        for f in fichiers
    ) / (1024 * 1024)  # Conversion en Mo

    print(f"   {nom_zip} extrait avec succès — {taille_extraite:.1f} Mo\n")

    # Sauvegarder le chemin pour l'utiliser dans les étapes suivantes
    chemins_extraits[annee] = dossier_destination

print("\n Extraction terminée pour tous les fichiers.")


 Début de l'extraction des archives ZIP...

 Cette opération peut prendre plusieurs minutes selon la taille des fichiers.

    2017_02.zip déjà extrait — on passe.
    2018_02.zip déjà extrait — on passe.
    2019_02.zip déjà extrait — on passe.
    2020_02.zip déjà extrait — on passe.
    2021_02.zip déjà extrait — on passe.
    2022_02.zip déjà extrait — on passe.
    2023_02.zip déjà extrait — on passe.
    2024_02.zip déjà extrait — on passe.
    2025_02.zip déjà extrait — on passe.
    2026_02.zip déjà extrait — on passe.

 Extraction terminée pour tous les fichiers.


In [ ]:
#  Vérification de la structure extraite

# On vérifie que l'extraction s'est bien passée en affichant
# l'arborescence des dossiers créés.

print("\n Structure des dossiers après extraction :\n")

for annee in sorted(chemins_extraits.keys()):

    dossier = chemins_extraits[annee]
    print(f"   Année {annee} : {dossier}")

    # Lister le contenu de premier niveau (les dossiers .SAFE)
    try:
        contenu_dossier = os.listdir(dossier)
        for element in contenu_dossier[:3]:  # Afficher les 3 premiers éléments
            print(f"      └── {element}")
        if len(contenu_dossier) > 3:
            print(f"      └── ... ({len(contenu_dossier)} éléments au total)")
    except Exception as e:
        print(f"   Erreur lors de la lecture : {e}")




 Structure des dossiers après extraction :

   Année 2017 : /content/sentinel-2_extraits/2017
      └── S2A_MSIL2A_20170228T092021_N0500_R093_T33PVQ_20230926T044000.SAFE
   Année 2018 : /content/sentinel-2_extraits/2018
      └── S2B_MSIL2A_20180218T092029_N0500_R093_T33PVQ_20230903T010656.SAFE
   Année 2019 : /content/sentinel-2_extraits/2019
      └── S2B_MSIL2A_20190223T092029_N0500_R093_T33PVQ_20221126T162219.SAFE
   Année 2020 : /content/sentinel-2_extraits/2020
      └── S2B_MSIL2A_20220227T092029_N0510_R093_T33PVQ_20240518T194404.SAFE
   Année 2021 : /content/sentinel-2_extraits/2021
      └── S2A_MSIL2A_20210227T092031_N0500_R093_T33PVQ_20230524T055517.SAFE
   Année 2022 : /content/sentinel-2_extraits/2022
      └── S2B_MSIL2A_20230222T092029_N0510_R093_T33PVQ_20240815T023209.SAFE
   Année 2023 : /content/sentinel-2_extraits/2023
      └── S2B_MSIL2A_20230222T092029_N0510_R093_T33PVQ_20240815T023209.SAFE
   Année 2024 : /content/sentinel-2_extraits/2024
      └── S2B_MSIL2A_20

In [ ]:
#  Recherche automatique des bandes dans tous les dossiers

# On parcourt tous les dossiers extraits pour trouver les fichiers
# correspondant aux bandes B02, B03, B04 et B08 de chaque année.



print("\n🔍 Recherche des bandes spectrales dans tous les dossiers extraits...\n")

bandes_par_annee = {}   # Dictionnaire final : année → bandes → chemin

for annee in sorted(chemins_extraits.keys()):

    dossier_annee = chemins_extraits[annee]
    bandes_par_annee[annee] = {}

    # Parcourir récursivement tous les fichiers du dossier
    # os.walk() descend dans tous les sous-dossiers
    for racine, sous_dossiers, fichiers in os.walk(dossier_annee):

        for fichier in fichiers:

            # On cherche uniquement les .jp2 dans R10m
            if "R10m" in racine and fichier.endswith(".jp2"):

                chemin_fichier = os.path.join(racine, fichier)

                # Identifier la bande à partir du nom du fichier
                for bande in ["B02", "B03", "B04", "B08"]:
                    if f"_{bande}_" in fichier:
                        bandes_par_annee[annee][bande] = chemin_fichier

    # Afficher le résultat pour cette année
    bandes_trouvees = list(bandes_par_annee[annee].keys())
    print(f"   {annee} : {len(bandes_trouvees)} bande(s) trouvée(s) → {bandes_trouvees}")


🔍 Recherche des bandes spectrales dans tous les dossiers extraits...

   2017 : 4 bande(s) trouvée(s) → ['B04', 'B03', 'B08', 'B02']
   2018 : 4 bande(s) trouvée(s) → ['B08', 'B04', 'B02', 'B03']
   2019 : 4 bande(s) trouvée(s) → ['B08', 'B03', 'B04', 'B02']
   2020 : 4 bande(s) trouvée(s) → ['B02', 'B03', 'B08', 'B04']
   2021 : 4 bande(s) trouvée(s) → ['B02', 'B03', 'B04', 'B08']
   2022 : 4 bande(s) trouvée(s) → ['B02', 'B03', 'B08', 'B04']
   2023 : 4 bande(s) trouvée(s) → ['B02', 'B03', 'B08', 'B04']
   2024 : 4 bande(s) trouvée(s) → ['B08', 'B02', 'B04', 'B03']
   2025 : 4 bande(s) trouvée(s) → ['B08', 'B03', 'B04', 'B02']
   2026 : 4 bande(s) trouvée(s) → ['B04', 'B02', 'B08', 'B03']


In [ ]:

# LECTURE DES BANDES SPECTRALES ET CRÉATION DES COMPOSITIONS RGB

# BLOC 1 : Installation et importation des bibliothèques
# rasterio  : lecture des images satellitaires géoréférencées
# numpy     : manipulation des tableaux numériques (matrices)
# matplotlib: création des graphiques et visualisations


!pip install rasterio -q

import rasterio                        # Lecture des fichiers .jp2 et .tif
import numpy as np                     # Calculs sur les tableaux numériques
import matplotlib.pyplot as plt        # Création des visualisations
import matplotlib.gridspec as gridspec # Organisation avancée des figures
import warnings                        # Gestion des avertissements Python

# On désactive les avertissements mineurs pour garder l'affichage propre
warnings.filterwarnings('ignore')

print(" Bibliothèques importées avec succès.")
print(f"   Rasterio version : {rasterio.__version__}")
print(f"   NumPy version    : {np.__version__}")

 Bibliothèques importées avec succès.
   Rasterio version : 1.5.0
   NumPy version    : 2.0.2


In [ ]:
#  Fonction de lecture d'une bande spectrale


def lire_bande(chemin_fichier):
    """
    Lit un fichier image satellitaire (.jp2 ou .tif) et retourne
    son contenu sous forme de tableau NumPy 2D.


    """

    # Ouverture du fichier en lecture avec rasterio
    # 'r' signifie mode lecture (read)
    with rasterio.open(chemin_fichier) as source:

        # read(1) lit la première bande de l'image
        # (les fichiers Sentinel-2 individuels n'ont qu'une seule bande)
        # Le résultat est un tableau 2D : tableau[ligne][colonne] = valeur_pixel
        tableau = source.read(1)

        # Les métadonnées contiennent les informations géographiques :
        # système de coordonnées, résolution, coin supérieur gauche, etc.
        meta = source.meta

    return tableau, meta


# Test de la fonction sur la première année disponible
annee_test = "2017"   # On commence par tester avec 2017

print(f"\n Test de lecture des bandes pour l'année {annee_test} :\n")

for nom_bande in ["B02", "B03", "B04", "B08"]:

    # Récupérer le chemin du fichier depuis le dictionnaire créé à l'étape 2
    chemin = bandes_par_annee[annee_test][nom_bande]

    # Lire la bande
    tableau, meta = lire_bande(chemin)

    # Afficher les informations sur cette bande
    print(f"   Bande {nom_bande} :")
    print(f"      Dimensions   : {tableau.shape[0]} lignes × {tableau.shape[1]} colonnes")
    print(f"      Valeur min   : {tableau.min()}")
    print(f"      Valeur max   : {tableau.max()}")
    print(f"      Valeur moy.  : {tableau.mean():.1f}")
    print(f"      Type données : {tableau.dtype}")
    print()




 Test de lecture des bandes pour l'année 2017 :

   Bande B02 :
      Dimensions   : 10980 lignes × 10980 colonnes
      Valeur min   : 0
      Valeur max   : 20048
      Valeur moy.  : 1922.8
      Type données : uint16

   Bande B03 :
      Dimensions   : 10980 lignes × 10980 colonnes
      Valeur min   : 0
      Valeur max   : 15208
      Valeur moy.  : 2355.3
      Type données : uint16

   Bande B04 :
      Dimensions   : 10980 lignes × 10980 colonnes
      Valeur min   : 0
      Valeur max   : 14864
      Valeur moy.  : 2666.4
      Type données : uint16

   Bande B08 :
      Dimensions   : 10980 lignes × 10980 colonnes
      Valeur min   : 0
      Valeur max   : 17024
      Valeur moy.  : 3808.7
      Type données : uint16



In [ ]:
#  Fonction de normalisation pour la visualisation


def normaliser_bande(tableau, percentile_bas=2, percentile_haut=98):
    """
    Normalise les valeurs d'une bande entre 0 et 1 pour la visualisation.
    Utilise les percentiles pour éviter l'effet des valeurs aberrantes.

        tableau normalisé entre 0.0 et 1.0
    """

    # Calculer les seuils bas et haut
    # np.percentile(arr, 2) donne la valeur en dessous de laquelle
    # se trouvent 2% des pixels
    valeur_basse = np.percentile(tableau, percentile_bas)
    valeur_haute = np.percentile(tableau, percentile_haut)

    # Tronquer les valeurs entre les deux seuils
    # np.clip() force toutes les valeurs à rester dans [valeur_basse, valeur_haute]
    tableau_tronque = np.clip(tableau, valeur_basse, valeur_haute)

    # Normalisation linéaire entre 0 et 1
    # Si valeur_haute == valeur_basse (image uniforme), on retourne des zéros
    if valeur_haute == valeur_basse:
        return np.zeros_like(tableau, dtype=np.float32)

    tableau_normalise = (tableau_tronque - valeur_basse) / (valeur_haute - valeur_basse)

    # Conversion en float32 (nombres décimaux sur 32 bits)
    # float64 serait plus précis mais prend deux fois plus de mémoire
    return tableau_normalise.astype(np.float32)


print(" Fonction de normalisation définie.")


 Fonction de normalisation définie.


In [ ]:
#  Fonction de création d'une composition RGB



def creer_rgb(bandes_dict, annee):
    """
    Crée une composition RGB à partir des bandes B04, B03, B02.

    Paramètres :
        bandes_dict : dict — dictionnaire {bande: chemin_fichier}
        annee       : str  — année de l'image (pour affichage)

    Retourne :
        image_rgb : np.array 3D (hauteur, largeur, 3) normalisé entre 0 et 1
    """

    # Lire chaque bande nécessaire pour le RGB
    rouge, _ = lire_bande(bandes_dict["B04"])   # B04 = Canal Rouge
    vert,  _ = lire_bande(bandes_dict["B03"])   # B03 = Canal Vert
    bleu,  _ = lire_bande(bandes_dict["B02"])   # B02 = Canal Bleu

    # Normaliser chaque bande individuellement
    rouge_norm = normaliser_bande(rouge)
    vert_norm  = normaliser_bande(vert)
    bleu_norm  = normaliser_bande(bleu)

    # Empiler les trois bandes pour créer l'image RGB 3D
    # np.dstack() = "depth stack" : empile en profondeur
    # Résultat : tableau de forme (lignes, colonnes, 3)
    image_rgb = np.dstack([rouge_norm, vert_norm, bleu_norm])

    return image_rgb


print("Fonction de création RGB définie.")


Fonction de création RGB définie.


In [ ]:
#  Visualisation de quelques années pour vérification
import gc
import sys
import matplotlib.pyplot as plt

# Années à visualiser pour la vérification
annees_a_visualiser = ["2017", "2020", "2023", "2026"]

print("\n Création des visualisations RGB par année...\n")

# Titres des colonnes
titres_colonnes = [
    "Composition RGB\n(vraies couleurs)",
    "B02 — Bleu",
    "B03 — Vert",
    "B04 — Rouge",
    "B08 — Proche infrarouge"
]

for idx, annee in enumerate(annees_a_visualiser):

    # Vérifier que cette année est disponible
    if annee not in bandes_par_annee:
        print(f"   Année {annee} non disponible, on passe.")
        continue

    print(f"   Traitement et affichage de l'année {annee}...")

    # ASTUCE RAM : Créer une petite figure UNIQUEMENT pour l'année en cours (1 ligne, 5 colonnes)
    fig, axes = plt.subplots(nrows=1, ncols=5, figsize=(20, 5))

    #  Colonne 0 : Composition RGB
    image_rgb = creer_rgb(bandes_par_annee[annee], annee)

    ax = axes[0]
    ax.imshow(image_rgb)
    ax.set_title(f"{annee}\n{titres_colonnes[0]}", fontsize=10, fontweight='bold')
    ax.axis('off')

    #  Colonnes 1 à 4 : Bandes individuelles en niveaux de gris
    for j, nom_bande in enumerate(["B02", "B03", "B04", "B08"]):

        # Lire et normaliser la bande
        tableau, _ = lire_bande(bandes_par_annee[annee][nom_bande])
        tableau_norm = normaliser_bande(tableau)

        ax = axes[j + 1]
        ax.imshow(tableau_norm, cmap='gray', vmin=0, vmax=1)
        ax.set_title(f"{annee}\n{titres_colonnes[j+1]}", fontsize=10)
        ax.axis('off')

    # Sauvegarde individuelle de l'année dans Google Drive (optionnel mais recommandé pour garder une trace)
    plt.tight_layout()
    chemin_figure_annee = f"/content/drive/MyDrive/verification_rgb_{annee}.png"
    plt.savefig(chemin_figure_annee, dpi=120, bbox_inches='tight')

    # Affichage immédiat de l'année en cours
    plt.show()

    # --- NETTOYAGE CRUCIAL DE LA RAM ---
    plt.close(fig)       # Ferme la figure pour libérer la mémoire graphique
    del image_rgb, fig, axes  # Supprime les grosses variables de l'année en cours
    gc.collect()         # Force Colab à vider sa mémoire RAM
    # -----------------------------------

print("\n Traitement terminé avec succès sans planter la RAM !")


 Création des visualisations RGB par année...

   Traitement et affichage de l'année 2017...
